# Finite-Size Scaling at the Critical Point

At $\lambda_c = 1$, the Cluster-Ising Model is a $c = 3/2$ CFT.

**Critical exponents**: $\nu = z = 1$, $\beta = 3/8$, $\alpha = 0$

**Scaling predictions**:
- Energy gap: $\Delta(L) \sim L^{-z}$
- Order parameter: $m_y(L) \sim L^{-\beta/\nu}$ near $\lambda_c$
- Entanglement entropy: $S_{L/2} = \frac{c}{6} \log_2 L + \text{const}$ (OBC)

This notebook extracts these exponents from ED and DMRG data.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import curve_fit
import sys, os
sys.path.insert(0, os.path.abspath('..'))

%matplotlib inline
plt.rcParams.update({'font.size': 12, 'figure.figsize': (10, 6)})

## 0. Parameters

In [ ]:
# === System sizes for ED ===
L_ed_list = [6, 8, 10, 12, 14]      # ED (keep max <= 16)
bc = 'open'                           # boundary condition

# === System sizes for DMRG ===
run_dmrg = True                       # set False to skip
L_dmrg_list = [20, 40, 60, 80]       # DMRG system sizes (L=100 is very slow at critical)
chi_max = 120                          # bond dimension (needs high chi at c=3/2 critical)

# === Lambda values around critical point ===
lam_c = 1.0
lam_near = np.array([0.7, 0.8, 0.9, 0.95, 1.0, 1.05, 1.1, 1.2, 1.3])

print(f"ED sizes: {L_ed_list}")
if run_dmrg:
    print(f"DMRG sizes: {L_dmrg_list}, chi={chi_max}")

## 1. Energy gap scaling: $\Delta(L) \sim L^{-z}$ at $\lambda = 1$

In [ ]:
from cluster_ising.solvers.ed_solver import run_ed

# ED: compute gap at lambda=1 for various L
gaps_ed = {}
for L in L_ed_list:
    result = run_ed({'L': L, 'lam': lam_c, 'bc': bc}, n_states=4)
    evals = result.metadata['eigenvalues']
    gaps_ed[L] = evals[1] - evals[0]
    print(f"L={L:3d}: Delta = {gaps_ed[L]:.6f}")

# Fit: Delta = a * L^{-z}
L_arr = np.array(list(gaps_ed.keys()), dtype=float)
gap_arr = np.array(list(gaps_ed.values()))

def power_law(L, a, z):
    return a * L**(-z)

popt, pcov = curve_fit(power_law, L_arr, gap_arr, p0=[3.0, 1.0])
z_fit = popt[1]
z_err = np.sqrt(pcov[1, 1])
print(f"\nFitted z = {z_fit:.4f} +/- {z_err:.4f} (exact: z=1)")

In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))

ax.loglog(L_arr, gap_arr, 'bo', ms=8, label='ED data')
L_fit = np.linspace(min(L_arr), max(L_arr)*1.5, 100)
ax.loglog(L_fit, power_law(L_fit, *popt), 'b--',
          label=f'Fit: $\\Delta \\propto L^{{-{z_fit:.3f}}}$')
ax.loglog(L_fit, popt[0] * L_fit**(-1.0), 'k:', alpha=0.5,
          label='$L^{-1}$ (exact $z=1$)')

ax.set_xlabel('$L$')
ax.set_ylabel('$\\Delta(L)$')
ax.set_title(f'Energy gap scaling at $\\lambda_c = 1$ (z = {z_fit:.3f})')
ax.legend()
plt.tight_layout()
plt.show()

## 2. Entanglement entropy: $S = \frac{c}{6}\log_2 L + \text{const}$

In [ ]:
from cluster_ising.observables.entanglement import half_chain_entropy, fit_central_charge

# ED: half-chain entropy at lambda=1 for various L
S_ed = {}
for L in L_ed_list:
    result = run_ed({'L': L, 'lam': lam_c, 'bc': bc}, n_states=1)
    psi = result.state
    S_ed[L] = half_chain_entropy(psi, L, 'vector')
    print(f"L={L:3d}: S_half = {S_ed[L]:.6f}")

# Fit central charge
L_arr_s = np.array(list(S_ed.keys()), dtype=float)
S_arr = np.array(list(S_ed.values()))
c_fit, const_fit, c_err = fit_central_charge(L_arr_s, S_arr, bc=bc)
print(f"\nFitted c = {c_fit:.4f} +/- {c_err:.4f} (exact: c = 1.5)")

In [ ]:
import warnings
warnings.filterwarnings('ignore', category=UserWarning)

# Add DMRG data if enabled
S_dmrg = {}
if run_dmrg:
    from cluster_ising.solvers.dmrg_solver import run_dmrg as _run_dmrg
    from tqdm.auto import tqdm

    dmrg_params = {
        'trunc_params': {'chi_max': chi_max, 'svd_min': 1e-10},
        'mixer': True,
        'mixer_params': {'amplitude': 1e-5, 'decay': 1.5, 'disable_after': 30},
        'max_sweeps': 60,
        'max_E_err': 1e-12,
        'max_S_err': 1e-8,
    }

    for L in tqdm(L_dmrg_list, desc='DMRG entropy'):
        model_params = {
            'L': L, 'lam': lam_c,
            'bc_MPS': 'finite', 'conserve': 'parity',
        }
        result = _run_dmrg(model_params, dmrg_params)
        psi = result.state
        S_dmrg[L] = half_chain_entropy(psi, L, 'mps')
        print(f"L={L:3d}: S_half = {S_dmrg[L]:.6f}, chi_max_used = {max(psi.chi)}")

    # Combined fit (ED + DMRG)
    all_L = np.array(list(S_ed.keys()) + list(S_dmrg.keys()), dtype=float)
    all_S = np.array(list(S_ed.values()) + list(S_dmrg.values()))
    c_comb, const_comb, c_comb_err = fit_central_charge(all_L, all_S, bc=bc)
    print(f"\nCombined fit c = {c_comb:.4f} +/- {c_comb_err:.4f}")

In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))

ax.plot(np.log2(L_arr_s), S_arr, 'bo', ms=8, label='ED')

if S_dmrg:
    L_d = np.array(list(S_dmrg.keys()), dtype=float)
    S_d = np.array(list(S_dmrg.values()))
    ax.plot(np.log2(L_d), S_d, 'r^', ms=8, label='DMRG')
    # Use combined fit
    c_plot, const_plot = c_comb, const_comb
    L_all_max = max(max(L_arr_s), max(L_d))
else:
    c_plot, const_plot = c_fit, const_fit
    L_all_max = max(L_arr_s)

log2_L_fit = np.linspace(np.log2(min(L_arr_s)), np.log2(L_all_max*1.2), 50)
ax.plot(log2_L_fit, (c_plot / 6) * log2_L_fit + const_plot, 'k--',
        label=f'Fit: $c = {c_plot:.3f}$')
ax.plot(log2_L_fit, (1.5 / 6) * log2_L_fit + const_plot, ':', color='gray',
        alpha=0.5, label='$c = 3/2$ (exact)')

ax.set_xlabel('$\\log_2 L$')
ax.set_ylabel('$S_{L/2}$')
ax.set_title(f'Central charge extraction at $\\lambda_c = 1$')
ax.legend()
plt.tight_layout()
plt.savefig('central_charge.png', dpi=150, bbox_inches='tight')
plt.show()

## 3. Order parameter scaling: $m_y \sim |\lambda - 1|^{\beta}$ with $\beta = 3/8$

In [ ]:
from cluster_ising.observables.order_parameters import staggered_magnetization_y
from cluster_ising.models.exact_solution import staggered_magnetization

# ED: m_y vs lambda near critical point, for different L
my_fss = {}  # my_fss[L] = array of m_y values

for L in L_ed_list:
    my_vals = []
    for lam in lam_near:
        result = run_ed({'L': L, 'lam': lam, 'bc': bc}, n_states=1)
        my_vals.append(staggered_magnetization_y(result.state, L, 'vector'))
    my_fss[L] = np.array(my_vals)
    print(f"L={L}: m_y at lam=1.3 -> {my_fss[L][-1]:.5f}")

# Exact (thermodynamic limit)
my_exact_near = np.array([staggered_magnetization(l) for l in lam_near])

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# (a) m_y vs lambda for different L
ax = axes[0]
for L in L_ed_list:
    ax.plot(lam_near, my_fss[L], 'o-', ms=4, label=f'L={L}')
ax.plot(lam_near, my_exact_near, 'k-', lw=2, label='Exact ($N\\to\\infty$)')
ax.axvline(1.0, color='gray', ls='--', alpha=0.5)
ax.set_xlabel('$\\lambda$')
ax.set_ylabel('$m_y$')
ax.set_title('(a) Staggered magnetization near $\\lambda_c$')
ax.legend(fontsize=8)

# (b) m_y at fixed lambda > 1 vs L (log-log for scaling)
ax = axes[1]
lam_test = 1.1  # slightly into Ising phase
idx_test = np.argmin(np.abs(lam_near - lam_test))
my_at_lam = np.array([my_fss[L][idx_test] for L in L_ed_list])
L_arr_fss = np.array(L_ed_list, dtype=float)

ax.plot(L_arr_fss, my_at_lam, 'bo', ms=8)

# Fit m_y(L) = a * L^{-beta/nu} + m_y_inf
# For lambda slightly > 1, m_y should approach a finite value
# Near critical: m_y(L, lam_c) ~ L^{-beta/nu}
idx_crit = np.argmin(np.abs(lam_near - 1.0))
my_at_crit = np.array([my_fss[L][idx_crit] for L in L_ed_list])

# Fit at critical point
mask = my_at_crit > 1e-10
if np.sum(mask) >= 2:
    log_L = np.log(L_arr_fss[mask])
    log_m = np.log(my_at_crit[mask])
    coeffs = np.polyfit(log_L, log_m, 1)
    beta_nu = -coeffs[0]
    ax.loglog(L_arr_fss, my_at_crit, 'rs', ms=8, label=f'$\\lambda = 1.0$')
    L_fit = np.linspace(min(L_arr_fss), max(L_arr_fss), 50)
    ax.loglog(L_fit, np.exp(coeffs[1]) * L_fit**coeffs[0], 'r--',
             label=f'$L^{{-{beta_nu:.3f}}}$ (expect $3/8 = 0.375$)')

ax.loglog(L_arr_fss, my_at_lam, 'bo', ms=8, label=f'$\\lambda = {lam_test}$')
ax.set_xlabel('$L$')
ax.set_ylabel('$m_y(L)$')
ax.set_title(f'(b) Order parameter scaling')
ax.legend(fontsize=9)

fig.tight_layout()
plt.savefig('fss_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

## 4. Entanglement entropy profile $S(l)$ at $\lambda_c = 1$

In [ ]:
from cluster_ising.observables.entanglement import block_entropies, calabrese_cardy_entropy

fig, ax = plt.subplots(figsize=(8, 5))

for L in [10, 12, 14]:
    result = run_ed({'L': L, 'lam': lam_c, 'bc': bc}, n_states=1)
    S_profile = block_entropies(result.state, L, 'vector')
    bonds = np.arange(1, L)
    ax.plot(bonds, S_profile, 'o-', ms=4, label=f'ED L={L}')

    # Calabrese-Cardy prediction
    S_cc = calabrese_cardy_entropy(bonds, L, c=1.5, const=S_profile[L//2 - 1] -
        calabrese_cardy_entropy(L//2, L, c=1.5, bc=bc), bc=bc)
    ax.plot(bonds, S_cc, '--', alpha=0.5)

ax.set_xlabel('Bond position $l$')
ax.set_ylabel('$S(l)$')
ax.set_title(f'Block entanglement entropy at $\\lambda_c = 1$ ($c = 3/2$)')
ax.legend()
plt.tight_layout()
plt.show()